<a href="https://colab.research.google.com/github/hernon33/Project_Board/blob/main/Epic3_Tasks_Aidan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import math
from scipy import stats
import statsmodels.api as sm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# ============================
# Load dataset
# ============================

df = pd.read_csv("Initial_Modeling_Dataset.csv")

# convert bool to int if needed
for c in df.columns:
    if df[c].dtype == bool:
        df[c] = df[c].astype(int)

# ============================
# 1. NUMERIC DRIVER TESTS
# ============================

num_cols = [
    "satisfaction_level",
    "last_evaluation",
    "number_project",
    "average_montly_hours",
    "tenure_years"
]

num_results = []

for c in num_cols:

    leavers = df[df["attrition_flag"] == 1][c]
    stayers = df[df["attrition_flag"] == 0][c]

    t_stat, p_val = stats.ttest_ind(
        leavers,
        stayers,
        equal_var=False
    )

    # Cohen's d
    n1 = len(leavers)
    n0 = len(stayers)

    s1 = leavers.std(ddof=1)
    s0 = stayers.std(ddof=1)

    pooled = math.sqrt(
        ((n1 - 1) * s1**2 + (n0 - 1) * s0**2)
        / (n1 + n0 - 2)
    )

    cohens_d = (
        leavers.mean() - stayers.mean()
    ) / pooled

    num_results.append({
        "variable": c,
        "mean_leavers": leavers.mean(),
        "mean_stayers": stayers.mean(),
        "cohens_d": cohens_d,
        "p_value": p_val
    })

numeric_results = pd.DataFrame(num_results)

numeric_results.to_csv(
    "epic3_numeric_driver_tests.csv",
    index=False
)

# ============================
# 2. REBUILD CATEGORICAL VALUES
# ============================

salary_cat = np.select(
    [
        df["salary_low"] == 1,
        df["salary_high"] == 1
    ],
    [
        "low",
        "high"
    ],
    default="medium"
)

tenure_band_cat = np.select(
    [
        df["tenure_band_0-2"] == 1,
        df["tenure_band_6-10"] == 1
    ],
    [
        "0-2",
        "6-10"
    ],
    default="3-5"
)

dept_cols = [
    c for c in df.columns
    if c.startswith("department_")
]

department_cat = []

for _, row in df[dept_cols].iterrows():

    ones = [
        c.replace("department_", "")
        for c, v in row.items()
        if v == 1
    ]

    if len(ones) == 0:
        department_cat.append("sales")
    else:
        department_cat.append(ones[0])

df["salary_cat"] = salary_cat
df["tenure_band_cat"] = tenure_band_cat
df["department_cat"] = department_cat

# ============================
# 3. CRAMERS V
# ============================

def cramers_v(ct):

    chi2 = stats.chi2_contingency(ct)[0]

    n = ct.sum().sum()

    r, k = ct.shape

    phi2 = chi2 / n

    phi2corr = max(
        0,
        phi2 - ((k - 1) * (r - 1)) / (n - 1)
    )

    rcorr = r - ((r - 1)**2) / (n - 1)

    kcorr = k - ((k - 1)**2) / (n - 1)

    return np.sqrt(
        phi2corr / min((kcorr - 1), (rcorr - 1))
    )

# ============================
# 4. CATEGORICAL TESTS
# ============================

categorical_tests = []

cat_cols = [
    "salary_cat",
    "tenure_band_cat",
    "department_cat",
    "promotion_last_5years",
    "work_accident",
    "high_hours_flag",
    "low_satisfaction_flag",
    "early_tenure_flag"
]

for c in cat_cols:

    ct = pd.crosstab(
        df[c],
        df["attrition_flag"]
    )

    chi2, p_val, _, _ = stats.chi2_contingency(ct)

    cv = cramers_v(ct)

    rates = ct[1] / ct.sum(axis=1)

    for level, rate in rates.items():

        categorical_tests.append({
            "variable": c,
            "level": level,
            "attrition_rate": rate,
            "cramers_v": cv,
            "p_value": p_val
        })

categorical_results = pd.DataFrame(
    categorical_tests
)

categorical_results.to_csv(
    "epic3_categorical_driver_tests.csv",
    index=False
)

# ============================
# 5. LOGISTIC REGRESSION
# ============================

predictors = [
    "last_evaluation",
    "number_project",
    "tenure_years",
    "work_accident",
    "promotion_last_5years",
    "high_hours_flag",
    "low_satisfaction_flag",
    "salary_low",
    "salary_high",
    "department_IT",
    "department_RandD",
    "department_accounting",
    "department_hr",
    "department_management",
    "department_marketing",
    "department_product_mng",
    "department_support",
    "department_technical"
]

X = sm.add_constant(
    df[predictors]
)

y = df["attrition_flag"]

logit_model = sm.Logit(
    y,
    X
).fit()

conf = logit_model.conf_int()

logit_results = pd.DataFrame({

    "driver":
        logit_model.params.index,

    "coefficient":
        logit_model.params.values,

    "odds_ratio":
        np.exp(
            logit_model.params.values
        ),

    "ci_low":
        np.exp(
            conf[0].values
        ),

    "ci_high":
        np.exp(
            conf[1].values
        ),

    "p_value":
        logit_model.pvalues.values
})

logit_results.to_csv(
    "epic3_logistic_effect_sizes.csv",
    index=False
)

# ============================
# 6. RANDOM FOREST
# ============================

rf_features = [

    c for c in df.columns

    if c not in [

        "attrition_flag",
        "time_spend_company",
        "salary_cat",
        "tenure_band_cat",
        "department_cat"
    ]
]

Xrf = df[rf_features]

yrf = df["attrition_flag"]

X_train, X_test, y_train, y_test = train_test_split(

    Xrf,
    yrf,

    test_size=0.3,
    random_state=42,
    stratify=yrf
)

rf = RandomForestClassifier(

    n_estimators=300,

    random_state=42,

    n_jobs=-1,

    class_weight="balanced_subsample"
)

rf.fit(
    X_train,
    y_train
)

rf_pred = rf.predict_proba(
    X_test
)[:, 1]

rf_auc = roc_auc_score(
    y_test,
    rf_pred
)

rf_results = pd.DataFrame({

    "driver":
        rf_features,

    "rf_importance":
        rf.feature_importances_
})

rf_results.to_csv(
    "epic3_random_forest_importance.csv",
    index=False
)

print("Done")
print("AUC =", rf_auc)

Optimization terminated successfully.
         Current function value: 0.384488
         Iterations 7
Done
AUC = 0.9925257164205237
